In [1]:
import pandas as pd
import geohash2

In [2]:
ais_data_df=pd.read_csv("AIS_data_small.csv")
ships_small_df=pd.read_csv("ships_small.csv")
radio_signatures_df=pd.read_csv("radio_signatures_small.csv")

In [3]:
ais_data_df['geohash'] = ais_data_df.apply(
    lambda row: geohash2.encode(row['latitude'], row['longitude'], precision=8), 
    axis=1
)
ais_data_df = ais_data_df.drop(['latitude', 'longitude'], axis=1)

In [41]:
df_merged = ais_data_df.merge(
    ships_small_df[['mmsi', 'name', 'type', 'flag', 'destination']], 
    on='mmsi', 
    how='left'          # 'left', 'inner', 'right', ou 'outer'
)
df_merged = df_merged.merge(radio_signatures_df, on='mmsi', how='left')

In [42]:
df_merged.columns

Index(['mmsi', 'timestamp_x', 'speed', 'course', 'status', 'ais_active',
       'geohash', 'name', 'type', 'flag', 'destination', 'signature_id',
       'frequency', 'bandwidth', 'modulation', 'power', 'timestamp_y',
       'location_lat', 'location_lon', 'signal_strength'],
      dtype='str')

In [43]:
# ============================================================
# PIPELINE COMPLET ROBUSTE CATBOOST
# ============================================================

import os
import joblib
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

# ============================================================
# 1. LOAD DATA
# ============================================================

df = df_merged

# ============================================================
# 2. TIMESTAMPS
# ============================================================

timestamp_cols = ["timestamp_x", "timestamp_y"]

for col in timestamp_cols:

    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

for col in timestamp_cols:

    df[f"{col}_year"] = df[col].dt.year
    df[f"{col}_month"] = df[col].dt.month
    df[f"{col}_day"] = df[col].dt.day
    df[f"{col}_hour"] = df[col].dt.hour
    df[f"{col}_minute"] = df[col].dt.minute
    df[f"{col}_weekday"] = df[col].dt.weekday

# Delta temps
df["timestamp_diff_sec"] = (
    df["timestamp_y"] - df["timestamp_x"]
).dt.total_seconds()

# Drop timestamps bruts
df.drop(columns=timestamp_cols, inplace=True)

# ============================================================
# 4. MISSING VALUES
# ============================================================

num_cols = df.select_dtypes(
    include=["int64", "float64"]
).columns

cat_cols = df.select_dtypes(
    include=["object", "bool"]
).columns

# Fill numériques
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill catégories
for col in cat_cols:
    df[col] = df[col].fillna("UNKNOWN")

# ============================================================
# 5. TARGETS
# ============================================================

targets = [
    "geohash",
    "name",
    "type",
    "flag",
    "destination"
]

# ============================================================
# 6. FEATURES
# ============================================================

X = df.drop(columns=targets)

cat_features = X.select_dtypes(
    include=["object", "bool"]
).columns.tolist()

# ============================================================
# 7. SAVE PREPROCESSORS
# ============================================================

os.makedirs("modeles", exist_ok=True)
os.makedirs("preprocessors", exist_ok=True)

joblib.dump(
    X.columns.tolist(),
    "preprocessors/features.joblib"
)

joblib.dump(
    cat_features,
    "preprocessors/cat_features.joblib"
)

# ============================================================
# 8. TRAINING
# ============================================================

results = {}

for target in targets:

    print("\n================================================")
    print(f"TARGET : {target}")
    print("================================================")

    # ========================================================
    # TARGET
    # ========================================================

    y = df[target].astype(str)

    # Label encoding
    le = LabelEncoder()

    y_encoded = le.fit_transform(y)

    # Save encoder
    joblib.dump(
        le,
        f"preprocessors/label_encoder_{target}.joblib"
    )

    # ========================================================
    # ANALYSE CLASSES
    # ========================================================

    class_counts = pd.Series(y_encoded).value_counts()

    print("Nombre classes :", len(class_counts))
    print("Min samples classe :", class_counts.min())

    # ========================================================
    # CAS 1 : classes suffisantes
    # ========================================================

    if class_counts.min() >= 2:

        print("Split avec stratify")

        X_used = X
        y_used = y_encoded

        stratify_value = y_used

    # ========================================================
    # CAS 2 : classes rares
    # ========================================================

    else:

        print(
            "Classes rares détectées -> "
            "suppression classes uniques"
        )

        valid_classes = class_counts[
            class_counts >= 2
        ].index

        mask = pd.Series(y_encoded).isin(
            valid_classes
        )

        X_used = X[mask]
        y_used = y_encoded[mask]

        print(
            "Samples restants :",
            X_used.shape[0]
        )

        # Si dataset vide
        if X_used.shape[0] < 10:

            print(
                f"SKIP {target} : "
                "pas assez de données"
            )

            continue

        stratify_value = y_used

    # ========================================================
    # SPLIT
    # ========================================================

    X_train, X_test, y_train, y_test = train_test_split(
        X_used,
        y_used,
        test_size=0.2,
        random_state=42,
        stratify=stratify_value
    )

    # ========================================================
    # MODEL
    # ========================================================

    model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=8,
        loss_function="MultiClass",
        eval_metric="Accuracy",
        verbose=100
    )

    # ========================================================
    # TRAIN
    # ========================================================

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_test, y_test),
        use_best_model=True
    )

    # ========================================================
    # PREDICTIONS
    # ========================================================

    preds = model.predict(X_test)

    acc = accuracy_score(
        y_test,
        preds
    )

    print(f"Accuracy {target}: {acc:.4f}")

    results[target] = acc

    # ========================================================
    # SAVE MODEL
    # ========================================================

    model_path = (
        f"modeles/catboost_{target}.joblib"
    )

    joblib.dump(model, model_path)

    print("Sauvegardé :", model_path)

# ============================================================
# 9. FINAL RESULTS
# ============================================================

print("\n================================================")
print("RESULTATS")
print("================================================")

for k, v in results.items():
    print(f"{k}: {v:.4f}")


TARGET : geohash
Nombre classes : 20
Min samples classe : 1
Classes rares détectées -> suppression classes uniques
Samples restants : 0
SKIP geohash : pas assez de données

TARGET : name
Nombre classes : 20
Min samples classe : 1
Classes rares détectées -> suppression classes uniques
Samples restants : 0
SKIP name : pas assez de données

TARGET : type
Nombre classes : 2
Min samples classe : 1
Classes rares détectées -> suppression classes uniques
Samples restants : 19


CatBoostError: catboost/private/libs/target/target_converter.cpp:404: Target contains only one unique value